In [ ]:
%pylab inline
import matplotlib.pyplot as plt
from ipywidgets import *

# Gömbtükör – gömbi tükör sugarainak rajzolása

A gömbtükör görbületi sugara $R$, fókusztávolsága $f = R/2$.

A tengellyel párhuzamos beeső sugarak visszaverődés után a fókuszponton mennek át (paraxiális közelítésben).
Nagy $y$ esetén a gömbtükör-aberráció látható: a szélső sugarak nem pontosan a fókuszba konvergálnak.

In [ ]:
# ################################################################
# Gömbtükör: a visszavert sugár iránya pontos geometriával
# ################################################################

# A gömbtükör: homorú tükör, középpontja (R, 0), sugara R
# A tükör felszínének egyenlete: (x - R)^2 + y^2 = R^2
#   --> x_m = R - sqrt(R^2 - y^2)  (a bal oldali ág)
#
# Beeső sugár: párhuzamos az x tengellyel, magassága y0
# Visszavert sugár iránya: pontos visszaverési törvény alapján

def tukor_pont(y0, R):
    """A tükörpont koordinatája adott y0 magasságban."""
    if abs(y0) > R:
        return None
    xm = R - sqrt(R**2 - y0**2)
    return array([xm, y0])

def visszavert_irany(y0, R):
    """
    Visszavert sugár irányvektora párhuzamos beeső sugárnál (irány: (1, 0)).
    A normális a tükörponton átmenő, a gömb középpontja felé mutató vektor.
    """
    pm = tukor_pont(y0, R)
    if pm is None:
        return None
    # Normális vektor: a középpont (R, 0) irányába
    n = array([R, 0]) - pm          # = (sqrt(R^2 - y0^2), -y0)
    n = n / linalg.norm(n)           # normált
    d = array([1.0, 0.0])            # beeső sugár iránya
    # Visszavert irány: d_r = d - 2*(d·n)*n
    dr = d - 2 * dot(d, n) * n
    return dr

def fokuszpont(R):
    """Paraxiális fókusztávolság: f = R/2."""
    return R / 2

In [ ]:
# ################################################################
# Gömbtükör rajzolása adott R és sugárszám mellett
# ################################################################

def gorbe_tukor_rajz(R, nn):
    """
    Gömbtükör és a beeső/visszavert sugarak rajzolása.

    R  = görbületi sugár
    nn = beeső párhuzamos sugarak száma
    """
    figsize(10, 8)
    ax = subplot(1, 1, 1, aspect='equal')

    # A tükör felszínének rajzolása
    y_max = min(0.95 * R, R)        # a tükör y kiterjedése
    y_tukor = linspace(-y_max, y_max, 300)
    x_tukor = R - sqrt(R**2 - y_tukor**2)
    plot(x_tukor, y_tukor, color='k', lw=3, label='tükör')

    # Fókuszpont és középpont jelölése
    f = fokuszpont(R)
    plot(f, 0, 'ko', markersize=8)
    plot(R, 0, 'k+', markersize=12, markeredgewidth=2)
    ax.annotate(r'$F$  ($f=R/2$)', xy=(f, 0), xytext=(f + 0.05 * R, 0.08 * R), fontsize=13)
    ax.annotate(r'$C$  (középpont)', xy=(R, 0), xytext=(R + 0.03 * R, 0.08 * R), fontsize=13)

    # Optikai tengely
    plot([-0.2 * R, 1.5 * R], [0, 0], 'k--', lw=0.8, alpha=0.5)

    # Sugarak rajzolása
    hossz_be = 1.2 * R     # beeső sugár hossza
    hossz_ki = 1.5 * R     # visszavert sugár hossza

    y_values = linspace(-y_max * 0.95, y_max * 0.95, nn)

    for y0 in y_values:
        pm = tukor_pont(y0, R)
        dr = visszavert_irany(y0, R)
        if pm is None or dr is None:
            continue

        # Beeső sugár: (pm[0] - hossz_be, y0) --> pm
        plot([pm[0] - hossz_be, pm[0]], [y0, y0], color='b', lw=1.2)

        # Visszavert sugár: pm --> pm + hossz_ki * dr
        xveg = pm[0] + hossz_ki * dr[0]
        yveg = pm[1] + hossz_ki * dr[1]
        plot([pm[0], xveg], [pm[1], yveg], color='r', lw=1.2)

    xlim(-0.2 * R, 2.2 * R)
    ylim(-1.3 * y_max, 1.3 * y_max)
    xlabel('x', fontsize=13)
    ylabel('y', fontsize=13)
    title('Gömbtükör  –  görbületi sugár $R = %.2f$,  fókusztávolság $f = R/2 = %.2f$' % (R, f),
          fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ################################################################
# Interaktív megjelenítés
# R  = görbületi sugár
# nn = a beeső párhuzamos sugarak száma
# ################################################################

@interact(R=(0.5, 5.0, 0.1), nn=(1, 21, 1))
def play(R=2.0, nn=9):
    gorbe_tukor_rajz(R, nn)
    plt.show()

In [ ]:
# ################################################################
# Gömbtükör-aberráció: a szélső sugarak metszetpontjainak rajzolása
# az y0 impaktparaméter függvényében
# ################################################################

def metszéspont_x(y0, R):
    """
    A visszavert sugár metszi az optikai tengelyt (y=0) egy x pontban.
    Visszaadja ezt az x értéket.
    """
    pm = tukor_pont(y0, R)
    dr = visszavert_irany(y0, R)
    if pm is None or dr is None or abs(dr[1]) < 1e-12:
        return None
    # Paraméteres egyenes: (pm[0] + t*dr[0], pm[1] + t*dr[1]) = (x, 0)
    t = -pm[1] / dr[1]
    x_met = pm[0] + t * dr[0]
    return x_met

R = 2.0
y_vals = linspace(0.01, 0.95 * R, 200)
x_met_vals = [metszéspont_x(y, R) for y in y_vals]
x_met_vals = array([x for x in x_met_vals if x is not None and x > 0])

figsize(8, 5)
plot(linspace(0.01, 0.95, len(x_met_vals)), x_met_vals, color='b')
axhline(y=fokuszpont(R), color='r', linestyle='--', label=r'$f = R/2 = %.2f$' % fokuszpont(R))
xlabel(r'$y_0 / R$  (normált impaktparaméter)', fontsize=13)
ylabel(r'Metszéspontja az optikai tengellyel  $x$', fontsize=13)
title('Gömbtükör-aberráció  ($R = %.1f$)' % R, fontsize=13)
legend(fontsize=12)
plt.tight_layout()
plt.show()
print('Paraxiális fókusztávolság: f = R/2 =', fokuszpont(R))